## 데이터 불러오기

In [ ]:
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import random
import os

import optuna
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.model_selection import cross_val_score
from optuna.samplers import TPESampler

from optuna import Trial

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import ExtraTreesRegressor
%matplotlib inline

pd.set_option('mode.chained_assignment',  None) # <==== 경고를 끈다

filename = 'data/train.csv'
data_train = pd.read_csv(filename)

filename = 'data/test.csv'
data_test = pd.read_csv(filename)

filename = 'data/sample_submission.csv'
submission = pd.read_csv(filename)

# 데이터 분석

### 데이터 target 분포 확인

In [ ]:
from collections import Counter

def print_mode(df, col):

  cnt = Counter(df[col])
  list_cnt = cnt.most_common(3)

  for idx, value in enumerate(list_cnt):

    print(f'{col}의 최빈값 {idx+1}순위 : {value[0]} & {value[-1]}개')

In [ ]:
def print_statistics(df, col):

  max = df['착과량(int)'].max()
  min = df['착과량(int)'].min()
  mean = df['착과량(int)'].mean()
  median = df['착과량(int)'].median()

  print(f'{col}의 최대값 : {max}')
  print(f'{col}의 최소값 : {min}')
  print(f'{col}의 평균값 : {mean}')
  print(f'{col}의 중앙값 : {median}')
  print_mode(df, col)

In [ ]:
def identify_hist(df, col):

  sns.histplot(data=df[col], kde=True)
  print_statistics(df, col)

# 데이터 전처리

In [ ]:
#학습, 정답데이터 분리
y_train = data_train['착과량(int)']
X_drop_list = ['ID']
X_train = data_train.drop(X_drop_list, axis = 1)
X_test = data_test.drop(["ID"], axis = 1)

In [ ]:
#feature selection
high_corr = data_train.corr().abs().sort_values(by='착과량(int)',ascending=False).iloc[:,:1]
features_name = high_corr[high_corr['착과량(int)']>0.9].index
features_name = list(features_name)
features_name.remove('착과량(int)')
X,y = X_train.drop(['착과량(int)'], axis=1) , X_train['착과량(int)']

X = X[features_name]
X_test = X_test[features_name]

### 파생변수 생성

# 모델 생성

In [ ]:
#seed 고정
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
seed_everything(42) # Seed 고정

### nmae score metric

In [ ]:
#base version
def NMAE(true, pred):
    mae = np.mean(np.abs(true - pred))
    score = mae / np.mean(np.abs(true))
    return score

#cross_val custom version
def NMAE_CV(clf, x, y):
    pred = clf.predict(x)
    mae = np.mean(np.abs(y - pred))
    score = mae / np.mean(np.abs(y))
    return score

### TabnetRegressor

In [ ]:
from pytorch_tabnet.tab_model import TabNetRegressor
import torch

In [ ]:
X_numpy = X.to_numpy()
y_numpy = y.to_numpy().reshape(-1, 1)
X_test_numpy = X_test.to_numpy()

In [ ]:
skf = StratifiedKFold(n_splits = 5, random_state = 42, shuffle = True)
skf = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
tab_pred = []
i = 0
tab_nmae = []

for tr_idx, val_idx in skf.split(X_numpy, y_numpy):
    
    tr_x, tr_y = X_numpy[tr_idx], y_numpy[tr_idx]
    val_x, val_y = X_numpy[val_idx], y_numpy[val_idx]

    tab = TabNetRegressor(verbose = 100,seed = 42,optimizer_fn=torch.optim.AdamW)
    tab.fit(tr_x, tr_y, eval_set = [(tr_x, tr_y), (val_x, val_y)], patience=100, max_epochs=2000, eval_metric = ['mae'])

    val_pred = tab.predict(val_x).astype(int)
    fold_nmae = NMAE(val_y, val_pred)
    tab_nmae.append(fold_nmae)
    print(f"{i + 1} Fold NMAE = {fold_nmae}")

    i += 1

    fold_pred = tab.predict(X_test_numpy)
    tab_pred.append(fold_pred)

print(f"\nAVG of NMAE = {np.mean(tab_nmae)}")

In [ ]:
tab_pred = np.mean(tab_pred,axis = 0)
tab_pred = pd.Series(tab_pred.flatten())
tab_pred = tab_pred.to_numpy()

### XGBRegressor

In [ ]:
sampler = TPESampler()

In [ ]:
def objective(trial):
    kf = KFold(n_splits = 5, random_state = 42, shuffle = True)
    #train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.3,random_state=42)
    param = {
        'lambda': trial.suggest_float('lambda', 1e-3, 0.1),
        'alpha': trial.suggest_float('alpha', 1e-3, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'learning_rate': trial.suggest_float('learning_rate',0.01, 0.05),
        'n_estimators': trial.suggest_int('n_estimators', 1000, 3000),
        'max_depth': trial.suggest_int('max_depth', 4,24),
        'min_child_weight': trial.suggest_int('min_child_weight', 2, 50),
    }
    model = XGBRegressor(**param)  
    """
    base optuna searching method
    model.fit(train_x,train_y,eval_set=[(test_x,test_y)],early_stopping_rounds=100, eval_metric='mae')
    preds = model.predict(test_x).astype(int)
    nmae = NMAE(test_y, preds)
    return nmae
    """
    #cross_val optuna searching method
    score = cross_val_score(model, X, y, cv = kf, scoring = NMAE_CV).mean()
    return score
          
study_xgb = optuna.create_study(
    direction='minimize',
    study_name = 'Xgboost Optuna', 
    sampler=sampler
)
study_xgb.optimize(objective, n_trials=100)

In [ ]:
study_xgb.best_params

In [ ]:
xgb_param = {
    'objective' : 'reg:squarederror',
    'tree_method' : 'gpu_hist',
    'predictor' : 'gpu_predictor'
}
xgb_best_params = study_xgb.best_params.copy()
xgb_best_params.update(xgb_param)
xgb_best_params

In [ ]:
#multi-kfold
"""
import xgboost as xgb
def custom_NMAE(y_pred, Dmatrix):
    true = Dmatrix.get_label()
    mae = np.mean(np.abs(true - y_pred))
    score = mae / np.mean(np.abs(true))
    return ('custom_NMAE', score)
"""
xgb_pred = []

kfold_list = [2, 3, 4, 5, 6, 10, 20]
for kfold in kfold_list:
    print(f"{kfold} Fold start")
    i = 0
    xgb_nmae = []
    kf = KFold(n_splits=kfold, random_state=42, shuffle=True)
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        tr_x, tr_y = X.iloc[tr_idx], y.iloc[tr_idx]
        val_x, val_y = X.iloc[val_idx], y.iloc[val_idx]
        """
        파이썬 래퍼 XGB 데이터 셋 변환
        train = xgb.DMatrix(tr_x, label=tr_y)
        valid = xgb.DMatrix(val_x, label=val_y)
        test = xgb.DMatrix(X_test)
        """
        #사이킷 런 래퍼 XGB 학습
        xgb = XGBRegressor(**xgb_best_params)
        xgb.fit(tr_x, tr_y, eval_set = [(val_x, val_y)], early_stopping_rounds = 100, verbose = 50, eval_metric = 'mae')       
        val_pred = xgb.predict(val_x).astype(int)
        fold_nmae = NMAE(val_y, val_pred)
        xgb_nmae.append(fold_nmae)
        print(f"{i + 1}/{kfold} Fold NMAE = {fold_nmae}")
        i += 1
        fold_pred = xgb.predict(X_test)
        xgb_pred.append(fold_pred)
        
        """
        파이썬 래퍼 XGB 학습
        model = xgb.train(**xgb_best_params, train, 
                          num_boost_round = 1000, 
                          early_stopping_rounds = 100,
                          verbose = 50,
                          evals=[(valid, 'valid')], 
                          feval=custom_NMAE)
        """

    print(f"\nAVG of NMAE = {np.mean(xgb_nmae)}")

In [ ]:
xgb_pred_sum = sum(xgb_pred)  
xgb_pred_sum /= len(xgb_pred)
xgb_pred_sum

### LGBMRegressor

In [ ]:
def objective(trial):
    kf = KFold(n_splits = 5, random_state = 42, shuffle = True)
    #train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.3,random_state=42)
    param = {'num_leaves': trial.suggest_int('num_leaves', 10, 400), 
            'max_depth': trial.suggest_int('max_depth', 4, 24), 
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.5), 
            'n_estimators': trial.suggest_int('n_estimators', 1000, 3000), 
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 50), 
            'subsample': trial.suggest_uniform('subsample', 0.7, 1.0), 
            'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.7, 1.0),
            'reg_alpha': trial.suggest_uniform('reg_alpha', 0.0, 1.0),
            'reg_lambda': trial.suggest_uniform('reg_lambda', 0.0, 1.0),
            'random_state': 42}
    model =LGBMRegressor(**param)  
    
    """
    model.fit(train_x,train_y,eval_set=[(test_x,test_y)],early_stopping_rounds=100,eval_metric='mae')
    preds = model.predict(test_x).astype(int)
    nmae = NMAE(test_y, preds)
    return nmae
    """
    
    score = cross_val_score(model, X, y, cv = kf, scoring = NMAE_CV).mean()
    return score
          
study_lgb = optuna.create_study(
    direction='minimize',
    study_name = 'LGBM Optuna', 
    sampler=sampler
)
study_lgb.optimize(objective, n_trials=100)

In [ ]:
study_lgb.best_params

In [ ]:
lgb_param = {
    'objective' : 'regression',
    'device' : 'gpu',
    'metric' : 'mae',
}
lgb_best_params = study_lgb.best_params.copy()
lgb_best_params.update(lgb_param)
lgb_best_params

In [ ]:
#multi-kfold
lgb_pred = []

kfold_list = [2, 3, 4, 5, 6, 10, 20]
for kfold in kfold_list:
    print(f"{kfold} Fold start")
    i = 0
    lgb_nmae = []
    kf = KFold(n_splits=kfold, random_state=42, shuffle=True)
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        tr_x, tr_y = X.iloc[tr_idx], y.iloc[tr_idx]
        val_x, val_y = X.iloc[val_idx], y.iloc[val_idx]

        lgb = LGBMRegressor(**lgb_param)
        lgb.fit(tr_x, tr_y, eval_set = [(val_x, val_y)], early_stopping_rounds = 100, verbose = 50, eval_metric = 'mae')
        val_pred = lgb.predict(val_x).astype(int)
        fold_nmae = NMAE(val_y, val_pred)
        lgb_nmae.append(fold_nmae)
        print(f"{i + 1}/{kfold} Fold NMAE = {fold_nmae}")
        i += 1
        fold_pred = lgb.predict(X_test)
        lgb_pred.append(fold_pred)

    print(f"\nAVG of NMAE = {np.mean(lgb_nmae)}")

In [ ]:
lgb_pred_sum = sum(lgb_pred)  
lgb_pred_sum /= len(lgb_pred)
lgb_pred_sum

### CatboostRegressor

In [ ]:
#multi-kfold
cat_pred = []

kfold_list = [2, 3, 4, 5, 6, 10, 20]
for kfold in kfold_list:
    print(f"{kfold} Fold start")
    i = 0
    cat_nmae = []
    kf = KFold(n_splits=kfold, random_state=42, shuffle=True)
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        tr_x, tr_y = X.iloc[tr_idx], y.iloc[tr_idx]
        val_x, val_y = X.iloc[val_idx], y.iloc[val_idx]

        cat = CatBoostRegressor(max_depth = 4, learning_rate = 0.01, use_best_model = True, iterations = 3000, eval_metric = 'MAE')
        cat.fit(tr_x, tr_y, eval_set = [(val_x, val_y)], early_stopping_rounds = 100, verbose = 50)
        val_pred = cat.predict(val_x).astype(int)
        fold_nmae = NMAE(val_y, val_pred)
        cat_nmae.append(fold_nmae)
        print(f"{i + 1}/{kfold} Fold NMAE = {fold_nmae}")
        i += 1
        fold_pred = cat.predict(X_test)
        cat_pred.append(fold_pred)

    print(f"\nAVG of NMAE = {np.mean(cat_nmae)}")

In [ ]:
cat_pred_sum = sum(cat_pred)  
cat_pred_sum /= len(cat_pred)
cat_pred_sum

## Ensemble

In [ ]:
# submission['착과량(int)'] = xgb_pred_sum*0.5 + lgb_pred_sum*0.3 + cat_pred_sum*0.2
#submission['착과량(int)'] = np.round(submission['착과량(int)']) #정수화

In [ ]:
# submission

### Submission

In [ ]:
# submission.to_csv('submissions/multi_kfold_submission.csv', index=False)